# Prefix Reuse and KMP


```{contents}
:local:
:depth: 2
```


```{index} Reuse a Prefix Only When It Is Also the Needed Suffix
```



```{index} Reuse a Prefix Only When It Is Also the Needed Suffix
```

## Reuse a Prefix Only When It Is Also the Needed Suffix



KMP remembers how much of a pattern has already matched. After a mismatch, some of that matched text may also be a valid beginning of the next candidate match. The prefix table identifies such borders: proper prefixes that are also suffixes. Proper excludes the complete string itself, otherwise fallback would make no progress.

For pattern ABAB, the longest proper border of the whole pattern is AB, length two. For its prefixes A, AB, ABA and ABAB, the corresponding border lengths are 0, 0, 1 and 2. These are properties of the pattern; preparation can be reused when searching different texts under the same matching policy.

| Pattern prefix | Longest proper border | Length |
| --- | --- | --- |
| A | Empty | 0 |
| AB | Empty | 0 |
| ABA | A | 1 |
| ABAB | AB | 2 |

Suppose the matcher has recognized ABA of pattern ABAB and the next text code unit is C rather than the expected B. The matched prefix ABA has border A, of length one. It need not pretend that all recognized text is unrelated to the pattern. Falling back to a valid border keeps a suffix that already matches a prefix. If the new comparison also fails, repeat fallback through shorter borders. The text position can stay at the current unconsumed code unit while the matched-prefix length decreases.

For related explanation, see the further reading below.[^development-reading]


```{index} Separate Two Progress Arguments
```



```{index} Separate Two Progress Arguments
```

## Separate Two Progress Arguments



During scanning, either the text position advances or the matched-prefix length decreases through valid fallback links. A decrease cannot happen indefinitely without intervening increases. This accounting explains a linear number of scan comparisons under the algorithm's model; the proof is not simply that the outer loop mentions each text position once. The inner fallback work must be accounted for too.

Preparation costs work proportional to the pattern length. Scanning costs proportional to text length, with output storage proportional to the number of reported matches. When the same pattern is searched in many texts, an owned prepared table can avoid repeating preparation. If the table is exposed for arbitrary mutation, callers can break the matcher invariant; ownership/copying rules from the object chapters remain relevant.

After a full match, fallback must preserve the possibility of overlapping results. Resetting the matched length blindly to zero can lose occurrences such as ABA at starts zero and two in ABABA. Empty patterns need their explicit boundary contract before normal prefix-table indexing, because there is no pattern position zero to compare.

Guided exercise: build the prefix lengths for AAAA and explain why the final border has length three, not four. Trace the all-match scan on AAAAA. Compare the start positions with the direct matcher, then compare complete preparation-plus-scan work for one text and for several texts. Equivalent answers come before performance claims.


```{index} Proper Prefixes Survive a Mismatch
```

## Proper Prefixes Survive a Mismatch




```{index} Proper Prefixes Survive a Mismatch
```

prefix[i] is the longest proper prefix of pattern[0..i] that is also its suffix. Proper excludes the whole segment. For ababaca, the table is 0,0,1,2,3,0,1. If q units matched and the next comparison fails, only borders of the matched prefix can remain possible: set q=prefix[q−1] and try again.

In [ ]:
using System;
using System.Linq;
using System.Collections.Generic;
using System.Numerics;

var matcher=new PatternScan("ababaca");
Console.WriteLine("prefix="+string.Join(",",matcher.Prefix));
Console.WriteLine($"preparation comparisons={matcher.PreparationComparisons}");
var snapshot=matcher.Prefix;snapshot[2]=99;
Console.WriteLine("owned prefix="+string.Join(",",matcher.Prefix));
public sealed class PatternScan
{
    public record Matches(int[] Positions,long Comparisons);
    private readonly string pattern;
    private readonly int[] prefix;
    public long PreparationComparisons { get; }
    public int[] Prefix => (int[])prefix.Clone();
    public PatternScan(string pattern)
    {
        ArgumentNullException.ThrowIfNull(pattern);this.pattern=pattern;
        prefix=new int[pattern.Length];long comparisons=0;int matched=0;
        for(int i=1;i<pattern.Length;i++)
        {
            while(true)
            {
                comparisons++;
                if(pattern[i]==pattern[matched]){prefix[i]=++matched;break;}
                if(matched==0)break;
                matched=prefix[matched-1];
            }
        }
        PreparationComparisons=comparisons;
    }
    // Ordinal UTF-16 code units; all overlapping matches, including empty boundaries.
    public Matches Search(string text)
    {
        ArgumentNullException.ThrowIfNull(text);
        if(pattern.Length==0)return new Matches(Enumerable.Range(0,text.Length+1).ToArray(),0);
        var positions=new List<int>();int matched=0;long comparisons=0;
        for(int i=0;i<text.Length;i++)
        {
            while(true)
            {
                comparisons++;
                if(text[i]==pattern[matched])
                {
                    matched++;
                    if(matched==pattern.Length){positions.Add(i-pattern.Length+1);matched=prefix[matched-1];}
                    break;
                }
                if(matched==0)break;
                matched=prefix[matched-1];
            }
        }
        return new Matches(positions.ToArray(),comparisons);
    }
    public static Matches Direct(string text,string pattern)
    {
        ArgumentNullException.ThrowIfNull(text);ArgumentNullException.ThrowIfNull(pattern);
        var positions=new List<int>();long comparisons=0;
        for(int start=0;start<=text.Length-pattern.Length;start++)
        {
            int offset=0;
            while(offset<pattern.Length)
            {comparisons++;if(text[start+offset]!=pattern[offset])break;offset++;}
            if(offset==pattern.Length)positions.Add(start);
        }
        return new Matches(positions.ToArray(),comparisons);
    }
}


Prefix construction maintains a valid border length for the already processed prefix. On mismatch, following the previous border chain finds the next possible border rather than resetting blindly or skipping a valid candidate. Successful comparisons increase q by at most one; fallback decreases it. Across all positions the total fallback work is linear in m. The owned table is prepared once and can be reused across multiple texts.

```{index} KMP Keeps the Text Moving Forward
```

## KMP Keeps the Text Moving Forward




```{index} KMP Keeps the Text Moving Forward
```

Before each comparison, q code units of the pattern equal the suffix of text already consumed. After a mismatch, fallback preserves that invariant without moving the text index backward. After a complete match, fall back to the pattern’s longest border so overlapping occurrences remain discoverable.

In [ ]:
using System;
using System.Linq;
using System.Collections.Generic;
using System.Numerics;

var matcher=new PatternScan("aaa");
foreach(string text in new[]{"aaaaa","baaab",""})
{
 var result=matcher.Search(text);
 Console.WriteLine($"text=[{text}], positions=[{string.Join(",",result.Positions)}], scan comparisons={result.Comparisons}");
}
Console.WriteLine("empty boundaries="+string.Join(",",new PatternScan("").Search("abc").Positions));
public sealed class PatternScan
{
    public record Matches(int[] Positions,long Comparisons);
    private readonly string pattern;
    private readonly int[] prefix;
    public long PreparationComparisons { get; }
    public int[] Prefix => (int[])prefix.Clone();
    public PatternScan(string pattern)
    {
        ArgumentNullException.ThrowIfNull(pattern);this.pattern=pattern;
        prefix=new int[pattern.Length];long comparisons=0;int matched=0;
        for(int i=1;i<pattern.Length;i++)
        {
            while(true)
            {
                comparisons++;
                if(pattern[i]==pattern[matched]){prefix[i]=++matched;break;}
                if(matched==0)break;
                matched=prefix[matched-1];
            }
        }
        PreparationComparisons=comparisons;
    }
    // Ordinal UTF-16 code units; all overlapping matches, including empty boundaries.
    public Matches Search(string text)
    {
        ArgumentNullException.ThrowIfNull(text);
        if(pattern.Length==0)return new Matches(Enumerable.Range(0,text.Length+1).ToArray(),0);
        var positions=new List<int>();int matched=0;long comparisons=0;
        for(int i=0;i<text.Length;i++)
        {
            while(true)
            {
                comparisons++;
                if(text[i]==pattern[matched])
                {
                    matched++;
                    if(matched==pattern.Length){positions.Add(i-pattern.Length+1);matched=prefix[matched-1];}
                    break;
                }
                if(matched==0)break;
                matched=prefix[matched-1];
            }
        }
        return new Matches(positions.ToArray(),comparisons);
    }
    public static Matches Direct(string text,string pattern)
    {
        ArgumentNullException.ThrowIfNull(text);ArgumentNullException.ThrowIfNull(pattern);
        var positions=new List<int>();long comparisons=0;
        for(int start=0;start<=text.Length-pattern.Length;start++)
        {
            int offset=0;
            while(offset<pattern.Length)
            {comparisons++;if(text[start+offset]!=pattern[offset])break;offset++;}
            if(offset==pattern.Length)positions.Add(start);
        }
        return new Matches(positions.ToArray(),comparisons);
    }
}


Both preprocessing and scan use one counted equality test per candidate attempt, avoiding a hidden duplicate comparison in a while condition followed by an if. Preprocessing is O(m), each scan O(n), and the table O(m) space plus O(r) result storage. Total fallback decreases are bounded by total increases in matched length, which gives the linear bound despite the nested loop. Preparing for each request loses the benefit of repeated-pattern reuse.

```{index} Measure a Complete Workload and Define Text Units
```

## Measure a Complete Workload and Define Text Units




```{index} Measure a Complete Workload and Define Text Units
```

On repetitive text, direct scanning can repeatedly compare long matching prefixes. Include preparation in the KMP total, and reuse the same prepared pattern when comparing several texts. Operation counts are evidence, not timings.

In [ ]:
using System;
using System.Linq;
using System.Collections.Generic;
using System.Numerics;

string text=new string('a',200)+"b",pattern=new string('a',20)+"b";
var direct=PatternScan.Direct(text,pattern);var matcher=new PatternScan(pattern);var kmp=matcher.Search(text);
if(!direct.Positions.SequenceEqual(kmp.Positions))throw new Exception("Matcher disagreement.");
Console.WriteLine($"positions={string.Join(",",kmp.Positions)}, direct={direct.Comparisons}, KMP prep={matcher.PreparationComparisons}, scan={kmp.Comparisons}, total={matcher.PreparationComparisons+kmp.Comparisons}");
string unicode="A😀B";
Console.WriteLine($"UTF-16 length={unicode.Length}, B index={new PatternScan("B").Search(unicode).Positions[0]}");
public sealed class PatternScan
{
    public record Matches(int[] Positions,long Comparisons);
    private readonly string pattern;
    private readonly int[] prefix;
    public long PreparationComparisons { get; }
    public int[] Prefix => (int[])prefix.Clone();
    public PatternScan(string pattern)
    {
        ArgumentNullException.ThrowIfNull(pattern);this.pattern=pattern;
        prefix=new int[pattern.Length];long comparisons=0;int matched=0;
        for(int i=1;i<pattern.Length;i++)
        {
            while(true)
            {
                comparisons++;
                if(pattern[i]==pattern[matched]){prefix[i]=++matched;break;}
                if(matched==0)break;
                matched=prefix[matched-1];
            }
        }
        PreparationComparisons=comparisons;
    }
    // Ordinal UTF-16 code units; all overlapping matches, including empty boundaries.
    public Matches Search(string text)
    {
        ArgumentNullException.ThrowIfNull(text);
        if(pattern.Length==0)return new Matches(Enumerable.Range(0,text.Length+1).ToArray(),0);
        var positions=new List<int>();int matched=0;long comparisons=0;
        for(int i=0;i<text.Length;i++)
        {
            while(true)
            {
                comparisons++;
                if(text[i]==pattern[matched])
                {
                    matched++;
                    if(matched==pattern.Length){positions.Add(i-pattern.Length+1);matched=prefix[matched-1];}
                    break;
                }
                if(matched==0)break;
                matched=prefix[matched-1];
            }
        }
        return new Matches(positions.ToArray(),comparisons);
    }
    public static Matches Direct(string text,string pattern)
    {
        ArgumentNullException.ThrowIfNull(text);ArgumentNullException.ThrowIfNull(pattern);
        var positions=new List<int>();long comparisons=0;
        for(int start=0;start<=text.Length-pattern.Length;start++)
        {
            int offset=0;
            while(offset<pattern.Length)
            {comparisons++;if(text[start+offset]!=pattern[offset])break;offset++;}
            if(offset==pattern.Length)positions.Add(start);
        }
        return new Matches(positions.ToArray(),comparisons);
    }
}


UTF-16 offsets count code units, not user-perceived characters; a supplementary emoji takes two units. Exact ordinal matching performs no case folding, culture rules, or Unicode normalization. Canonically equivalent spellings can therefore differ. For ordinary application code, use .NET search APIs with an explicit comparison policy.[^2602_shortest_paths_strings-1] Do not claim this educational implementation outperforms the library without a controlled benchmark.

```{index} Exercise
```

## Exercise




```{index} Exercise
```

Compare direct and KMP outputs for overlapping, empty, longer-than-text, and case-sensitive patterns. Check against ordinal library first-match results.

In [ ]:
using System;
using System.Linq;
using System.Collections.Generic;
using System.Numerics;

var cases=new[]{("aaaa","aa"),("abc",""),("ab","abcd"),("Audit","audit")};
// Your code starts here.
/* TODO: complete the task. */
// Your code ends here.

In [1]:
using System;
using System.Linq;
using System.Collections.Generic;
using System.Numerics;

// Solution
var cases=new[]{("aaaa","aa"),("abc",""),("ab","abcd"),("Audit","audit")};
foreach(var pair in cases)
{
 var direct=PatternScan.Direct(pair.Item1,pair.Item2);var kmp=new PatternScan(pair.Item2).Search(pair.Item1);
 int first=kmp.Positions.Length==0?-1:kmp.Positions[0];
 if(!direct.Positions.SequenceEqual(kmp.Positions)||first!=pair.Item1.IndexOf(pair.Item2,StringComparison.Ordinal))throw new Exception("Ordinal mismatch.");
 Console.WriteLine($"[{pair.Item1}]/[{pair.Item2}]: positions=[{string.Join(",",kmp.Positions)}], first={first}");
}
public sealed class PatternScan
{
    public record Matches(int[] Positions,long Comparisons);
    private readonly string pattern;
    private readonly int[] prefix;
    public long PreparationComparisons { get; }
    public int[] Prefix => (int[])prefix.Clone();
    public PatternScan(string pattern)
    {
        ArgumentNullException.ThrowIfNull(pattern);this.pattern=pattern;
        prefix=new int[pattern.Length];long comparisons=0;int matched=0;
        for(int i=1;i<pattern.Length;i++)
        {
            while(true)
            {
                comparisons++;
                if(pattern[i]==pattern[matched]){prefix[i]=++matched;break;}
                if(matched==0)break;
                matched=prefix[matched-1];
            }
        }
        PreparationComparisons=comparisons;
    }
    // Ordinal UTF-16 code units; all overlapping matches, including empty boundaries.
    public Matches Search(string text)
    {
        ArgumentNullException.ThrowIfNull(text);
        if(pattern.Length==0)return new Matches(Enumerable.Range(0,text.Length+1).ToArray(),0);
        var positions=new List<int>();int matched=0;long comparisons=0;
        for(int i=0;i<text.Length;i++)
        {
            while(true)
            {
                comparisons++;
                if(text[i]==pattern[matched])
                {
                    matched++;
                    if(matched==pattern.Length){positions.Add(i-pattern.Length+1);matched=prefix[matched-1];}
                    break;
                }
                if(matched==0)break;
                matched=prefix[matched-1];
            }
        }
        return new Matches(positions.ToArray(),comparisons);
    }
    public static Matches Direct(string text,string pattern)
    {
        ArgumentNullException.ThrowIfNull(text);ArgumentNullException.ThrowIfNull(pattern);
        var positions=new List<int>();long comparisons=0;
        for(int start=0;start<=text.Length-pattern.Length;start++)
        {
            int offset=0;
            while(offset<pattern.Length)
            {comparisons++;if(text[start+offset]!=pattern[offset])break;offset++;}
            if(offset==pattern.Length)positions.Add(start);
        }
        return new Matches(positions.ToArray(),comparisons);
    }
}


[aaaa]/[aa]: positions=[0,1,2], first=0
[abc]/[]: positions=[0,1,2,3], first=0
[ab]/[abcd]: positions=[], first=-1
[Audit]/[audit]: positions=[], first=-1


```{rubric} Footnotes
```


[^2602_shortest_paths_strings-1]: Microsoft documents the comparison overloads and empty-pattern behavior of [String.IndexOf](https://learn.microsoft.com/en-us/dotnet/api/system.string.indexof) and recommends making comparison choices explicit in [string best practices](https://learn.microsoft.com/en-us/dotnet/standard/base-types/best-practices-strings).
[^2602_shortest_paths_strings-2]: KMP’s linear bound concerns code-unit comparisons and pattern preparation; returning every match also requires output storage.

[^development-reading]: Further reading: [Sedgewick and Wayne, Strings](https://algs4.cs.princeton.edu/50strings/). The worked business scenarios and guided reasoning here are original teaching examples.
